# Module 3, Section 1a: The Data Flywheel in Code

This is the **same TechHub support agent** you built and evaluated in **Module 2**, now running in production. Section 1 walked through the production data flywheel by **clicking through the LangSmith UI**. This companion section does the **same flywheel entirely in code** with the LangSmith SDK — and ends by showing *exactly how a flagged conversation turns into an agent improvement*.

We run the agent locally with tracing on, score each **conversation (thread)** with an LLM-as-judge evaluator, route the unsatisfied ones into an annotation queue, curate a fix, and **re-measure**.

<div align="center">
    <img src="../../images/data_flywheel.png" width="700">
</div>

**Continuity with the rest of the workshop:**
- Same agent (`supervisor_hitl` + SQL agent) as **Module 2, Section 2**.
- Same LangSmith **project** as Module 2 (`langsmith-agent-lifecycle-workshop`). Our synthetic demo traffic is tagged `demo: flywheel-1a`.
- Same `correctness_evaluator` from Module 2 for the offline re-measure.

```
Production runs → satisfaction judge (online eval) → filter unsatisfied (automation)
   → annotation queue → diagnose root cause → golden dataset → offline eval → prompt fix → repeat
```

---

## 0. Setup

We trace every run to the **same project as Module 2** (`LANGSMITH_API_KEY` comes from `.env`). The plumbing — driving conversations, rendering transcripts, building thread links, polling for anchor runs — lives in `online_eval_development.py`, imported below so these cells stay focused on the flywheel.

In [ ]:
import os
import sys
from dotenv import load_dotenv

load_dotenv()

# Use the SAME project as Module 2 — inherited from your .env (LANGSMITH_PROJECT),
# so the flywheel runs on the agent you already know.
PROJECT = os.environ.get("LANGSMITH_PROJECT") or "langsmith-agent-lifecycle-workshop"
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = PROJECT

# Make the helper module importable regardless of the kernel's working dir.
# `config` is already importable (Module 2 relies on it), so we use it to find
# the repo root, then add this section's folder to the path.
import config

REPO_ROOT = os.path.dirname(os.path.abspath(config.__file__))
HELPERS_DIR = os.path.join(REPO_ROOT, "workshop_modules", "module_3")
if HELPERS_DIR not in sys.path:
    sys.path.insert(0, HELPERS_DIR)

from langsmith import Client
from online_eval_development import (
    run_conversations,
    collect_threads,
    format_conversation,
    thread_url,
    annotation_queue_url,
)

client = Client()
print(f"Tracing to project: {PROJECT}")

### Optional: reset before you start

Reruns reuse the queue and dataset by name. To start clean (e.g. before presenting), uncomment the cell below **first** — it deletes only this demo's queue and dataset, never the shared project. Left commented so "Run All" never wipes anything mid-demo.

In [ ]:
# # --- Optional: reset this demo before you start (uncomment to run) ---
# # Wipes only THIS demo's annotation queue and dataset (not the shared project).
# # Names match the queue in Section 4 and the dataset in Section 7.
# for q in client.list_annotation_queues(name="techhub-continuous-improvement"):
#     client.delete_annotation_queue(q.id)
# for d in client.list_datasets(dataset_name="techhub-online-eval-corrections"):
#     client.delete_dataset(dataset_id=d.id)
# print("Reset complete — starting fresh.")

## 1. Build the agent

The improved agent from **Module 2, Section 2**: the supervisor HITL graph with the **SQL agent** (real order data) and **docs agent** (RAG over specs + policies) — the same setup the production `supervisor_hitl_sql_agent` deployment runs.

<div align="center">
    <img src="../../images/supervisor_with_verification.png" width="700">
</div>

In [ ]:
from agents.supervisor_hitl_agent import create_supervisor_hitl_agent
from agents.sql_agent import create_sql_agent
from agents.docs_agent import create_docs_agent
from IPython.display import Image

# The same composition as the production deployment: supervisor HITL graph with
# the flexible SQL agent and the RAG docs agent. We keep a reference to the docs
# agent's default behavior — Section 7 will swap in an improved prompt and show
# the offline score move.
agent = create_supervisor_hitl_agent(
    db_agent=create_sql_agent(),
    docs_agent=create_docs_agent(),
)

display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

## 2. Generate "production" traffic — in parallel

We script the conversations instead of role-playing in Studio — reproducible, and we can seed a couple of **unsatisfied** ones so the flywheel has something to catch. The helper runs the threads concurrently and handles the HITL email interrupt for the verified conversation.

> **The failure to watch — the agent *recites* policy instead of *applying* it.** Our policy charges a **15% restocking fee** on opened electronics over $500 (it's in `return_policy.md`). But asked *"I'm returning my opened $1,199 laptop — full refund?"*, the agent answers **"yes, full refund, no fee"** — the opposite. It never connects the customer's specific case (opened + over $500) to the fee. That's a clean **prompt** problem: the agent isn't told to *apply* policy and disclose costs — exactly the "what do I change?" we'll fix with one prompt edit.

In [ ]:
# A mix of conversations. The first two should read as UNSATISFIED: the agent
# confidently quotes a FULL refund and misses the 15% restocking fee on opened
# electronics over $500. The last is SATISFIED (a verified order-status check,
# which also exercises the agent's HITL email verification).
DEMO_CONVERSATIONS = [
    # Unsatisfied: agent quotes a full refund and misses the over-$500 fee.
    [
        "If I return an opened laptop I paid $1,199 for because I changed my mind, do I get a full refund?",
        "Wait — your policy charges a 15% restocking fee on opened electronics over $500. You just told me I'd get everything back and never mentioned it. That's not okay.",
    ],
    # Unsatisfied: same mishandled fee, different item.
    [
        "If I return an opened monitor I paid $599 for because I changed my mind, will I get all my money back?",
        "That's not right. Your own policy says opened electronics over $500 carry a 15% restocking fee — you quoted me a full refund and never mentioned it.",
    ],
    # Satisfied: verified-account order status, fully resolved (exercises HITL).
    # Email is a real TechHub customer so identity verification passes.
    [
        "Can you check the status of my order ORD-2025-0027?",
        "william.davis@gmail.com",
        "Perfect, that's exactly what I needed to know — thanks so much for your help!",
    ],
]

# Run all conversations concurrently (independent threads) and flush traces.
thread_ids, messages_by_thread = await run_conversations(agent, DEMO_CONVERSATIONS)

for tid, convo in zip(thread_ids, DEMO_CONVERSATIONS):
    print(f"Ran thread {tid[:8]}  |  opened with: {convo[0][:55]}")
print(f"\nGenerated {len(thread_ids)} production conversations (traces flushed).")

### Open these conversations (threads) in LangSmith

A **thread** is one multi-turn conversation. Its **most recent root run is the anchor** — it carries the full message history and is the run a thread-level evaluator scores, so score, link, and review all point at one object. `collect_threads(...)` polls `client.read_thread(...)` until the run count settles.

In [ ]:
# Resolve project + tenant once so we can build thread (conversation) links.
project_id = client.read_project(project_name=PROJECT).id
TENANT_ID = client._get_tenant_id()

threads = collect_threads(client, project_id, thread_ids, messages_by_thread)

print(f"Found {len(threads)} conversations. Open the threads in LangSmith:\n")
for t in threads:
    convo = format_conversation(t["messages"])
    opener = convo.splitlines()[0] if convo.splitlines() else ""
    print(f"- {opener[:55]}\n  {thread_url(client, TENANT_ID, project_id, t['thread_id'])}\n")

## 3. The online evaluator — in code

An online evaluator is just an LLM-as-judge over production traffic that writes feedback back, and for satisfaction the unit is the **thread**. We load the same **User Satisfaction** template the UI offers (`USER_SATISFACTION_PROMPT` from **`openevals`**), wrap it with `create_llm_as_judge`, and record the verdict under the UI's `user_sentiment` key (`positive` / `negative`).

<div align="center">
    <img src="../../images/online_eval_process.png" width="800">
</div>

In [ ]:
from openevals import create_llm_as_judge
from openevals.prompts import USER_SATISFACTION_PROMPT
from config import DEFAULT_MODEL

# The off-the-shelf "User Satisfaction" template — identical to the one in the
# LangSmith UI's evaluator catalog. Its {outputs} slot is filled with the conversation.
print(USER_SATISFACTION_PROMPT)

# Feedback key matches the UI's Section 1 "User Sentiment" thread evaluator.
SENTIMENT_KEY = "user_sentiment"

sentiment_evaluator = create_llm_as_judge(
    prompt=USER_SATISFACTION_PROMPT,
    feedback_key=SENTIMENT_KEY,
    model=DEFAULT_MODEL,
)

Run the judge over each conversation and write the score back as **thread-level feedback**. There's no thread-targeting API, so we write to the thread's **anchor run** (where LangSmith surfaces multi-turn scores), keeping `user_sentiment` thread-only. This loop — judge, then `client.create_feedback(...)` — *is* what an online evaluator does on live traffic.

In [ ]:
judged = []
for t in threads:
    conversation = format_conversation(t["messages"])
    if not conversation.strip():
        continue

    # The template's {outputs} variable receives the full formatted conversation.
    result = sentiment_evaluator(outputs=conversation)
    # Map the satisfaction verdict onto the UI's categorical positive/negative.
    sentiment = "positive" if bool(result["score"]) else "negative"

    # Write to the thread's anchor run -> shows up as thread-level feedback.
    # Categorical feedback uses `value` (not `score`).
    client.create_feedback(
        t["anchor_run"].id,
        key=SENTIMENT_KEY,
        value=sentiment,
        comment=result.get("comment"),
    )

    judged.append(
        {
            "thread_id": t["thread_id"],
            "anchor_run_id": t["anchor_run"].id,  # most recent root run; carries full history
            "sentiment": sentiment,
            "reasoning": result.get("comment"),
            "url": thread_url(client, TENANT_ID, project_id, t["thread_id"]),
        }
    )
    opener = conversation.splitlines()[0] if conversation.splitlines() else ""
    print(f"{sentiment.upper():9} | {opener[:55]}")
    print(f"            see thread: {thread_url(client, TENANT_ID, project_id, t['thread_id'])}\n")

print(f"Scored {len(judged)} conversations and wrote thread-level {SENTIMENT_KEY} feedback.")

## 4. The annotation queue — in code

`create_annotation_queue` is a first-class SDK method, so the Section 1 queue setup is a single call.

> 💡 **This is the step LangSmith Engine automates.** Today a human reads the queue and decides what to fix. **Engine** does it continuously — clustering failures into named issues, diagnosing root cause against your code, and drafting the fix. We do it by hand here to make the loop explicit (see the final section).
>
> 🖼️ *[Image placeholder: the annotation queue with the two flagged restocking-fee conversations waiting for review.]*

In [ ]:
# Fixed name so reruns reuse one queue instead of piling up duplicates.
QUEUE_NAME = "techhub-continuous-improvement"

existing = list(client.list_annotation_queues(name=QUEUE_NAME, limit=1))
if existing:
    queue = existing[0]
    print(f"Reusing existing annotation queue: {queue.name}")
else:
    queue = client.create_annotation_queue(
        name=QUEUE_NAME,
        description="Production threads flagged by the online evaluator for human review.",
        rubric_instructions=(
            "Review conversations flagged as unsatisfied. Validate the failure, diagnose the "
            "root cause (prompt? docs? tool? logic?), and add a corrected example to the golden dataset."
        ),
    )
    print(f"Created annotation queue: {queue.name}")

print(f"Open it in LangSmith: {annotation_queue_url(client, queue)}")

## 5. The automation rule — in code

The UI rule is: *when `user_sentiment` is `negative`, add it to the queue.* With the judged threads in memory, that's a filter plus one call.

> Queues attach to **runs**, not threads — so we enqueue each unsatisfied conversation's **anchor run** (the run that already holds its `user_sentiment` and full history).

In [ ]:
unsatisfied = [j for j in judged if j["sentiment"] == "negative"]

if unsatisfied:
    client.add_runs_to_annotation_queue(
        queue.id, run_ids=[j["anchor_run_id"] for j in unsatisfied]
    )

print(f"Routed {len(unsatisfied)} unsatisfied conversation(s) to the annotation queue:\n")
for j in unsatisfied:
    print(f"- {(j['reasoning'] or '')[:90]}\n  conversation: {j['url']}\n")

print(f"Review them here: {annotation_queue_url(client, queue)}")

## 6. Diagnose & fix — hand it to your coding agent

The queue tells you *something's wrong*, not *what to change*. That's triage, not a fix — so who does the diagnosis?

You already have the tool: the **coding agent open in your editor right now** (Claude Code, Cursor, …). Paste the clustered failures in and ask it to find the root cause and propose a fix. It reasons over your actual codebase and routes the failure to the right lever:

| Lever | Symptom | Fix |
| --- | --- | --- |
| **Prompt** | Has the info/tools but behaves wrong (over-promises, skips steps) | Edit the system prompt |
| **Docs / RAG** | Right answer isn't retrievable (missing or buried) | Fix the document / chunking |
| **Tool** | Lacks a capability it implies it has | Add or fix a tool |
| **Logic / routing** | Wrong sub-agent ran, or the graph took a bad path | Fix the graph |

```
flagged threads  →  cluster failures  →  coding agent diagnoses + proposes fix  →  prove it (offline eval)
```

**Our case:** the fee is already in `return_policy.md`, so it's not a docs / tool / routing gap — the agent *recited* policy instead of *applying* it. That's a **prompt** gap. Below we prove the fix the Module 2 way: turn it into a failing test, change one line, watch it pass.

## 7. Close the loop: red → green with one prompt change

Turn "the user was unhappy" into a precise, re-runnable test — then watch one prompt line move it from **red → green**.

**Step 1 — curate the ground truth.** The reviewer (or your coding agent) writes what the agent *should* have said: the restocking fee applies, so it's not a full refund. These become regression examples.

In [ ]:
# A reviewer pulled the flagged conversations from the queue and wrote the
# CORRECT answer for an opened, over-$500 buyer's-remorse return: the restocking
# fee applies, so it is NOT a full refund. (The fee itself is in return_policy.md.)
curated = [
    {
        "question": "If I return an opened laptop I paid $1,199 for because I changed my mind, I'd get the full $1,199 back, right?",
        "answer": (
            "No — you would not get a full refund. An opened laptop is electronics over $500, so a "
            "buyer's-remorse return is subject to a restocking fee that's deducted from your refund, "
            "which means you won't get the entire purchase price back. The fee is waived only for "
            "defective items or items returned unopened."
        ),
    },
    {
        "question": "If I return an opened monitor I paid $599 for because I changed my mind, I'd get all $599 back, right?",
        "answer": (
            "No — not the full amount. The monitor is opened electronics over $500, so a buyer's-remorse "
            "return carries a restocking fee that's deducted from your refund; you won't receive the "
            "entire purchase price. The fee doesn't apply to defective or unopened items."
        ),
    },
]

# Fixed name so reruns reuse one dataset instead of piling up duplicates.
DATASET_NAME = "techhub-online-eval-corrections"

existing_ds = list(client.list_datasets(dataset_name=DATASET_NAME))
if existing_ds:
    dataset = existing_ds[0]
    print(f"Reusing dataset: {dataset.name}")
else:
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description="Reviewer-corrected answers for failures the satisfaction evaluator flagged.",
    )
    client.create_examples(
        dataset_id=dataset.id,
        inputs=[{"question": ex["question"]} for ex in curated],
        outputs=[
            {"messages": [{"role": "assistant", "content": ex["answer"]}]} for ex in curated
        ],
    )
    print(f"Created dataset with {len(curated)} curated example(s).")

print(f"Open the dataset: {dataset.url}")

**Step 2 — measure the *current* agent (expect red).** Same `correctness_evaluator` from Module 2. The current agent confirms a full refund and misses the fee, so it scores **incorrect** — and that red is now a durable test guarding the behavior.

In [ ]:
import os
import uuid
from evaluators import correctness_evaluator

# Quiet the tokenizers fork warning so the experiment output stays readable.
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def make_target(target_agent):
    """Build an offline target that asks the curated policy question against a
    given agent and returns its answer. These are general policy questions, so
    no identity verification interrupt fires — the target stays simple."""
    def target_function(inputs: dict) -> dict:
        config = {"configurable": {"thread_id": str(uuid.uuid4())}}
        result = target_agent.invoke(
            {"messages": [{"role": "user", "content": inputs["question"]}]}, config=config
        )
        return {"messages": [{"role": "assistant", "content": result["messages"][-1].content}]}

    return target_function


# BEFORE the fix — the current agent (default docs prompt). Expect RED: it quotes
# a full refund and misses the restocking fee the reference answer applies.
before = client.evaluate(
    make_target(agent),
    data=dataset.name,
    evaluators=[correctness_evaluator],
    experiment_prefix="online-eval-restocking-before",
    description="Before prompt fix: does the agent apply the over-$500 restocking fee?",
    max_concurrency=2,
)

**Step 3 — make the prompt change.** This is exactly the one-line fix your coding agent would propose: tell the docs agent to *apply* policy to the specific case and surface fees, not just recite it. We add it in code so the before/after stays reproducible.

In [ ]:
from agents.docs_agent import DOCS_AGENT_SYSTEM_PROMPT

# The one behavioral instruction the default prompt is missing.
APPLY_POLICY_INSTRUCTION = (
    "\n- When a customer describes a SPECIFIC situation (their item, price, timeframe, or condition), "
    "do not just recite the policy. Apply it to their case and state a clear verdict up front — whether "
    "they qualify and any fee that applies (such as a restocking fee on opened electronics over $500) — "
    "before giving the general details."
)
IMPROVED_DOCS_PROMPT = DOCS_AGENT_SYSTEM_PROMPT + APPLY_POLICY_INSTRUCTION

print("ADDED TO THE DOCS-AGENT PROMPT:")
print(APPLY_POLICY_INSTRUCTION.strip())

# Rebuild the SAME agent with the improved docs sub-agent. Nothing else changes.
agent_v2 = create_supervisor_hitl_agent(
    db_agent=create_sql_agent(),
    docs_agent=create_docs_agent(system_prompt=IMPROVED_DOCS_PROMPT),
)

**Step 4 — re-measure (expect green).** Re-run the *same* experiment against the improved agent. When `correctness` flips to green, you've closed the loop — and you can point at the exact prompt line that did it.

In [ ]:
# AFTER the fix — same dataset, same evaluator, improved prompt. Expect GREEN.
after = client.evaluate(
    make_target(agent_v2),
    data=dataset.name,
    evaluators=[correctness_evaluator],
    experiment_prefix="online-eval-restocking-after",
    description="After prompt fix: agent now applies the over-$500 restocking fee.",
    max_concurrency=2,
)

print("\nCompare the two experiments in LangSmith to see correctness move red -> green:")
print(f"  before: {before.experiment_name}")
print(f"  after:  {after.experiment_name}")
print("Open the dataset's Experiments tab to see them side by side:")
print(f"  {dataset.url}")

That red → green is the whole answer to *"how does the agent improve itself?"*:

```
flagged in production → diagnose root cause (prompt) → curated example → failing offline test
   → one prompt change → green → ship → keep watching
```

The agent didn't improve itself — the **flywheel** did: a production signal became a precise test, a human (or Engine) diagnosed the lever, and the fix is *proven* by a re-measurable experiment instead of a hunch.

## 8. LangSmith Engine — the automated version of this loop

Everything above you did **by hand** (with a little help from your coding agent). **[LangSmith Engine](https://docs.langchain.com/langsmith/engine)** runs that exact loop **continuously**, in-app — you move from *doing* the loop to *approving* it.

```
  YOU (this notebook):   detect → cluster → diagnose → fix → test → re-measure → ship
                              │
  ENGINE (automated):    watches all traces → clusters issues → diagnoses vs your codebase
                         → drafts a PR fix → proposes an online evaluator → pulls failing
                         traces into your dataset → you approve → reopens if it resurfaces
```

Same loop, no hand-cranking. The SDK flywheel you just built is the mental model; **Engine runs it for you** — currently in public beta ([announcement](https://www.langchain.com/blog/introducing-langsmith-engine) · [docs](https://docs.langchain.com/langsmith/engine)).

## Key takeaways

- **Satisfaction is a thread-level signal** — judged over the whole conversation, anchored on the thread's most recent root run (`client.read_thread(...)`).
- **Online evaluation is just a judge over conversations** — an `openevals` template plus `client.create_feedback(...)` on the thread anchor.
- **The queue is triage, not a fix.** The real work is **diagnosis** — and in 2026 that's an LLM/coding-agent routing the failure to the right lever (prompt / docs / tool / logic), proven with a re-measurable test. We flipped `correctness` red → green with one prompt line.
- **Engine automates the loop** — detect, cluster, diagnose, fix, add eval coverage, re-measure — leaving the human at the approval step.

```
Production runs → satisfaction judge (online eval) → filter unsatisfied (automation)
   → annotation queue → diagnose (coding agent) → golden dataset → offline eval (Module 2)
   → targeted prompt fix → re-measure → ship → repeat        [LangSmith Engine runs this for you]
```